In [162]:
from __future__ import annotations

import errno

import lmdb
import time
from pathlib import Path
from typing import Iterable, Iterator, Optional
import logging
import os
import re

# from scripts.regsetup import description

import record_pb2
from meta import Run, Dir, File

In [157]:
from db import Db

db_name = 'test-db'
path = Path(db_name)

readonly = True
readonly = False

create=False
# create=True

db = Db.open(path, readonly=readonly, create=create)
env = db.env

In [159]:
env, env.flags(), env.path(), env.info()

(<Environment at 0x2c44bfbca50>,
 {'subdir': True,
  'readonly': False,
  'metasync': True,
  'sync': True,
  'map_async': False,
  'readahead': True,
  'writemap': False,
  'meminit': False,
  'lock': True},
 'test-db',
 {'map_addr': 0,
  'map_size': 4294967296,
  'last_pgno': 7,
  'last_txnid': 12,
  'max_readers': 126,
  'num_readers': 1})

In [ ]:
with env.begin(
    write=False,
    # buffers= ?
) as txn:
    with txn.cursor() as cur:
        for key, value in cur.iternext():
            print(key, value)

In [156]:
env.close()

In [167]:
run_id = '0051'
run = Run(
    run_id,
    description='run from jupyter for development',
    specification='bla',
    start_time=1.234,
    end_time=1.234,
    duration=1.234,
    status='initialized')

def mk_run_key():
    return bytes(f'r:{run.id}', 'utf-8')

import record_pb2

def mk_run_rec():
    rec = record_pb2.RunRecord(
        schema_version=1,
        path=run.path,
        description=run.description,
        platform=run.specification,
        date_time=run.start_time,
        dur_secs=run.duration,
        status=run.status,
        num_dirs=run.num_dirs,
        num_files=run.num_files,
        # extra={'fld1': 'bla', 'fld2': 'san 64\r\nder 19'},
        error=None,
    )

    # return bytes(run.description, 'utf-8')
    rv = rec.SerializeToString()
    print(type(rv))
    return rv

key = mk_run_key()
value = mk_run_rec()
# value = msg.SerializeToString()

key, value


AttributeError: module 'record_pb2' has no attribute 'RunRecord'

In [152]:
with env.begin(write=True) as txn:
    txn.put(key, value)

txn

In [153]:
txn
# txn.stat()